In [2]:
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import time
from tqdm import tqdm

import uproot

In [3]:
parent = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/conf14"
file_path = "/mu3e_sort_run0001000.root"
in_dir = parent + file_path

train_dir = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/train"
val_dir = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/val"
test_dir = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/test"

In [4]:
def is_valid_file(path):
    path = Path(path)
    return path.is_file() and path.stat().st_size > 0

In [5]:
def load_real_event_data(path_in):
    print('Loading global hit and track data, compiling events...')
    track_fields = ["tid", "pdg", "vx", "vy", "vz", "vt", "px", "py", "pz"]
    hit_fields = ["tid", "hid", "det", "pdg", "x", "y", "z", "time", "edep", "px", "py", "pz"]
    
    path_in = Path(in_dir)
    
    with uproot.open(path_in) as file:
        # pd.DataFrame of track and hit data from mu3e tree - only with chosen fields
        tracks_flat = file['mu3e_mc_tracks'].arrays(track_fields, library="pd")
        hits_flat = file['mu3e_mchits'].arrays(hit_fields, library="pd")

        # jagged awkward array - each entry in jagged array = list of hit indices in mchits tree, each list = one event
        hit_mapping = file["mu3e"]["hit_mc_i"].array(library='ak')

    # dataframe tidying    
    global_track_df = tracks_flat.rename(columns={'tid': 'trackID'})
    global_hit_df = hits_flat.rename(columns={"tid": "trackID", "hid": "hitID"})
        
    return global_track_df, global_hit_df, hit_mapping

In [6]:
def root_to_parquet(in_dir: str, train_dir: str, val_dir: str, test_dir: str, event_limit: int=None):
    """
    Convert ROOT file to 2 parquet files (tracks and hits) with original eventID column.
    For hits without an associated event, assigns eventID : -1.
    """   
    path_in = Path(in_dir)
    
    path_train = Path(train_dir)
    path_val = Path(val_dir)
    path_test = Path(test_dir)
    
    path_train.mkdir(parents=True, exist_ok=True)
    path_val.mkdir(parents=True, exist_ok=True)
    path_test.mkdir(parents=True, exist_ok=True)
    
    ### loading hit and track dataframes ###
    global_track_df, global_hit_df, hit_mapping = load_real_event_data(path_in)
        
    ### mapping events ###
    print('Mapping events...')
    # flattening into one long numpy array of hit indices - all the hits in mchits that are included in monte carlo eventing
    flat_indices = np.asarray(ak.flatten(hit_mapping), dtype=np.int64)

    # creates array of eventIDs - maps onto flat_indices --- first N entries in event_ids = 0, maps to first N hit indices in flat_indices
    event_ids = np.repeat(np.arange(len(hit_mapping)), ak.num(hit_mapping))

    # above misses out all events that aren't accounted for in the mu3e tree - below assigns those events eventID = -1 -- makes easy to filter out later
    full_event_ids = np.full(len(global_hit_df), -1, dtype=np.int32)   # array of -1, length = total number of hits
    full_event_ids[flat_indices] = event_ids                           # applies eventIDs to all hit idx
    global_hit_df['eventID'] = full_event_ids                          # applied eventIDs to all hits in global hit dataframe

    tid_to_event = global_hit_df[global_hit_df['eventID'] != -1].set_index('trackID')['eventID']
    tid_to_event = tid_to_event[~tid_to_event.index.duplicated(keep='first')]
    global_track_df['eventID'] = global_track_df['trackID'].map(tid_to_event).fillna(-1).astype(int)
    
    ### applying event limit ###
    if event_limit:
        print(f'Limiting to {event_limit} compiled events...')
        global_hit_df = global_hit_df[global_hit_df["eventID"] < event_limit]
        global_track_df = global_track_df[global_track_df["eventID"] < event_limit]
        print('Ordering dataframes...')
    else:
        print('Ordering dataframes...')
        
    # sorting eventID and trackID
    global_hit_df = global_hit_df.sort_values(['eventID', 'trackID'])
    global_track_df = global_track_df.sort_values(['eventID', 'trackID'])
    
    sensor_hits = global_hit_df[global_hit_df['det']==10]
    valid_sensor_hits = sensor_hits[sensor_hits['eventID']!=-1].reset_index(drop=True)
    valid_global_tracks = global_track_df[global_track_df['eventID']!=-1].reset_index(drop=True)
    
    print('Separating into train, val, test groups...')
    train_hits = valid_sensor_hits[valid_sensor_hits["eventID"] < 38000]
    train_tracks = valid_global_tracks[valid_global_tracks["eventID"] < 38000]
    
    val_hits = valid_sensor_hits[(valid_sensor_hits["eventID"] >= 38000) & (valid_sensor_hits["eventID"] < 42500)]
    val_tracks = valid_global_tracks[(valid_global_tracks["eventID"] >= 38000) & (valid_global_tracks["eventID"] < 42500)]
    
    test_hits = valid_sensor_hits[valid_sensor_hits["eventID"] >= 42500]
    test_tracks = valid_global_tracks[valid_global_tracks["eventID"] >= 42500]

    # save to single parquet files
    print("Saving hits to parquet...")
    train_hits.to_parquet(path_train / "all_hits.parquet", index=False)
    val_hits.to_parquet(path_val / "all_hits.parquet", index=False)    
    test_hits.to_parquet(path_test / "all_hits.parquet", index=False)
    
    print("Saving tracks to parquet...")
    train_tracks.to_parquet(path_train / "all_tracks.parquet", index=False)
    val_tracks.to_parquet(path_val / "all_tracks.parquet", index=False)    
    test_tracks.to_parquet(path_test / "all_tracks.parquet", index=False)
    
    print("\n--- Done ---")
    print(f"Saved {len(train_tracks['eventID'].unique())} events to:")
    print(f"  - {path_train / 'all_tracks.parquet'}")
    print(f"  - {path_train / 'all_hits.parquet'}")
    print()
    print(f"Saved {len(val_tracks['eventID'].unique())} events to:")
    print(f"  - {path_val / 'all_tracks.parquet'}")
    print(f"  - {path_val / 'all_hits.parquet'}")
    print()
    print(f"Saved {len(test_tracks['eventID'].unique())} events to:")
    print(f"  - {path_test / 'all_tracks.parquet'}")
    print(f"  - {path_test / 'all_hits.parquet'}")
    
    return global_track_df, global_hit_df, valid_sensor_hits, valid_global_tracks

In [14]:
global_track_df, global_hit_df, valid_sensor_hits, valid_global_tracks = root_to_parquet(in_dir, train_dir, val_dir, test_dir, event_limit=None)

Loading global hit and track data, compiling events...
Mapping events...
Ordering dataframes...
Separating into train, val, test groups...
Saving hits to parquet...
Saving tracks to parquet...

--- Done ---
Saved 37978 events to:
  - /home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/train/all_tracks.parquet
  - /home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/train/all_hits.parquet

Saved 4499 events to:
  - /home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/val/all_tracks.parquet
  - /home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/val/all_hits.parquet

Saved 7496 events to:
  - /home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/test/all_tracks.parquet
  - /home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/test/all_hits.parquet


In [15]:
event_counts = global_hit_df.groupby("eventID").size()
valid_event_ids = event_counts[event_counts > 0].index
global_hit_df_2 = global_hit_df[global_hit_df["eventID"].isin(valid_event_ids)]

In [16]:
print(len(global_hit_df))
print(len(global_hit_df_2))

28522372
28522372


In [10]:
# global hit dataframe - all hits w/ eventID = -1 are removed
valid_global_hits = global_hit_df[global_hit_df['eventID']!=-1]

In [11]:
# all sensor hits w/ eventID = -1 are removed
valid_sensor_hits = sensor_hits[sensor_hits['eventID']!=-1]

# unique track IDs froms valid sensor hits 
valid_sensor_hits_track_ids = valid_sensor_hits['trackID'].unique()

# unique track IDs from ALL sensor hits, including eventID = -1
sensor_hits_track_ids = sensor_hits['trackID'].unique()

In [24]:
# global track dataframe - all tracks w/ eventID = -1] are removed
valid_global_tracks = global_track_df[global_track_df['eventID']!=-1]
# sensor track dataframe - all tracks corresponding to hits left over in hits after filtering out all det != 10
valid_sensor_tracks = global_track_df[global_track_df['trackID'].isin(sensor_hits_track_ids)]

In [25]:
valid_sensor_tracks_NaN = valid_sensor_tracks[valid_sensor_tracks['eventID']==-1]
valid_global_tracks_NaN = valid_global_tracks[valid_global_tracks['eventID']==-1]

In [32]:
print(f'valid sensor tracks has {len(valid_sensor_tracks)} tracks in it')
print(f'valid global tracks has {len(valid_global_tracks)} tracks in it')
print()
print(f'valid NaN sensor tracks has {len(valid_sensor_tracks_NaN)} tracks in it')
print(f'valid NaN global tracks has {len(valid_global_tracks_NaN)} tracks in it')
print()
print('where the freak is this discrepancy coming from...')
print('just gonna get rid of all eventID = -1 for now, only a 1000 tracks out of 380k')

valid sensor tracks has 380747 tracks in it
valid global tracks has 379680 tracks in it

valid NaN sensor tracks has 1067 tracks in it
valid NaN global tracks has 0 tracks in it

where the freak is this discrepancy coming from...
just gonna get rid of all eventID = -1 for now, only a 1000 tracks out of 380k


In [15]:
print(f"number of tracks that are assigned NaN eventID and are made up of sensor hits : {len(sensor_hits[sensor_hits['eventID']==-1]['trackID'].unique())}")

number of tracks that are assigned NaN eventID and are made up of sensor hits : 6762


## `EDA`

In [31]:
hit_df = valid_sensor_hits
track_df = valid_sensor_tracks

print(f"unique particle ids:   {np.unique(hit_df['pdg'].values).tolist()}")
print("")
print("--------------------------- hit stats ---------------------------")
print(f"electron hits:                  {(hit_df['pdg'].values == 11).sum()}")
print(f"positron hits:                  {(hit_df['pdg'].values == -11).sum()}")
print(f"antimuon hits:                  {(hit_df['pdg'].values == -13).sum()}")
print(f"photon hits:                    {(hit_df['pdg'].values == 22).sum()}")
print(f"proton hits:                    {(hit_df['pdg'].values == 2212).sum()}")
print(f"neutron hits:                   {(hit_df['pdg'].values == 2112).sum()}")
print(f"alpha particle hits:            {(hit_df['pdg'].values == 1000020040).sum()}")
print(f"Be nuclide hits:                {(hit_df['pdg'].values == 1000040080).sum()}")
print(f"C nuclide hits:                 {(hit_df['pdg'].values == 1000060120).sum()}")
print(f"Mg nuclide hits:                {(hit_df['pdg'].values == 1000120240).sum()}")
print(f"Si nuclide hits:                {(hit_df['pdg'].values == 1000140280).sum()}")
print(f"hits from unassigned events:    {(hit_df['eventID'].values == -1).sum()}")
print("")
print(f"total hits:                     {len(hit_df)}")
print("")
print("-------------------------- track stats --------------------------")
print(f"electron tracks:                {(track_df['pdg'].values == 11).sum()}")
print(f"positron tracks:                {(track_df['pdg'].values == -11).sum()}")
print(f"antimuon tracks:                {(track_df['pdg'].values == -13).sum()}")
print(f"photon tracks:                  {(track_df['pdg'].values == 22).sum()}")
print(f"proton tracks:                  {(track_df['pdg'].values == 2212).sum()}")
print(f"neutron tracks:                 {(track_df['pdg'].values == 2112).sum()}")
print(f"alpha particle tracks:          {(track_df['pdg'].values == 1000020040).sum()}")
print(f"Be nuclide tracks:              {(track_df['pdg'].values == 1000040080).sum()}")
print(f"C nuclide tracks:               {(track_df['pdg'].values == 1000060120).sum()}")
print(f"Mg nuclide tracks:              {(track_df['pdg'].values == 1000120240).sum()}")
print(f"Si nuclide tracks:              {(track_df['pdg'].values == 1000140280).sum()}")
print(f"tracks from unassigned events:  {(track_df['eventID'].values == -1).sum()}")
print("")
print(f"total tracks:                   {len(track_df)}")

unique particle ids:   [-13, -11, 11, 1000120240, 1000140280]

--------------------------- hit stats ---------------------------
electron hits:                  117792
positron hits:                  2860066
antimuon hits:                  87
photon hits:                    0
proton hits:                    0
neutron hits:                   0
alpha particle hits:            0
Be nuclide hits:                0
C nuclide hits:                 0
Mg nuclide hits:                1
Si nuclide hits:                1
hits from unassigned events:    0

total hits:                     2977947

-------------------------- track stats --------------------------
electron tracks:                18479
positron tracks:                362183
antimuon tracks:                82
photon tracks:                  0
proton tracks:                  0
neutron tracks:                 0
alpha particle tracks:          1
Be nuclide tracks:              0
C nuclide tracks:               0
Mg nuclide tracks:         

In [14]:
global_hit_df[global_hit_df['eventID']==1]

,trackID,hitID,det,pdg,x,y,z,time,edep,px,py,pz,eventID
25544444,12210,1,10,-11,22.649362,5.599192,9.556817,6784.545068,0.022293,15.430574,15.958057,-5.353213,1
25544445,12210,2,10,-11,27.465206,11.154845,7.540690,6784.570516,0.025496,12.727147,17.831881,-6.173153,1
25544446,12210,-6,10,-11,-72.711405,-0.148235,-71.212648,6785.739501,0.023373,16.961372,-12.788433,-5.031326,1
25544447,12210,3,10,-11,40.250046,59.869067,-5.240242,6784.747134,0.030487,-1.717291,21.532111,-5.076159,1
25544448,12210,-5,10,-11,-84.420288,10.543781,-67.633783,6785.685157,0.028040,14.183730,-16.016602,-4.588489,1
25544449,12210,4,10,-11,37.444487,76.858949,-9.490455,6784.806446,0.029338,-6.027892,20.627179,-5.313347,1
25544450,12842,1,10,-11,-21.617807,8.089587,-35.122070,6779.231905,0.031503,-15.738900,-3.407743,-17.054727,1
25544451,12842,2,10,-11,-29.106651,6.102998,-43.493010,6779.269977,0.035697,-14.934954,-5.212176,-16.763094,1
25544452,12842,3,10,-11,-64.981295,-31.266343,-101.915079,6779.535876,0.080501,-3.565553,-14.636217,-16.926800,1
25544453,12842,4,10,-11,-61.707365,-58.500788,-134.492128,6779.678792,0.100208,5.916451,-13.759871,-16.778381,1


In [1]:
valid_sensor_tracks

NameError: name 'valid_sensor_tracks' is not defined

## `PLOTTING`

In [ ]:
def plot_event(event):
    """
    Plot hit positions for a single event.
    
    Parameters:
    -----------
    event : awkward array
        Single event data containing hit position information
    """
    # Extract hit positions (adjust field names based on your data structure)
    x = ak.concatenate([event[key]['x'] for key in event.keys()])
    y = ak.concatenate([event[key]['y'] for key in event.keys()])
    z = ak.concatenate([event[key]['z'] for key in event.keys()])
    
    # Create figure with subplots
    fig, axes = plt.subplots(1, 3, figsize=(20,6.5))
    
    # XY projection
    axes[0].scatter(x, y, alpha=0.8, s=10)
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('y')
    axes[0].set_title('XY Projection')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-100,100)
    axes[0].set_ylim(-100,100)
    
    # XZ projection
    axes[1].scatter(z, x, alpha=0.8, s=10)
    axes[1].set_xlabel('z')
    axes[1].set_ylabel('x')
    axes[1].set_title('XZ Projection')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(-100,100)

    
    # YZ projection
    axes[2].scatter(z, y, alpha=0.8, s=10)
    axes[2].set_xlabel('z')
    axes[2].set_ylabel('y')
    axes[2].set_title('YZ Projection')
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim(-100,100)

    
    plt.tight_layout()
    plt.show()
    
    print(f"Number of hits: {len(x)}")